### Day-Night Image Classification

In this notebook, we will implement a DNN classifier to classify images taken by stationary cameras as day or night images. We will also establish a baseline to make sure our deep learning implementation outperforms a simple hand-picked feature-based classification.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

import pickle
import numpy as np
import keras
from keras import Model
from keras.layers import Input, Dense, Flatten
from sklearn.model_selection import train_test_split

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 6]

# Set the seed for reproducibility (python, numpy and the backend)
keras.utils.set_random_seed(1234)

#### Dataset Loading

We will use a subset of the [day-night image dataset](https://www.kaggle.com/datasets/stevemark/daynight-dataset?resource=download) (images taken by stationary cameras at different hours of the day). We have already downsampled the images to 64x64 pixels (RGB), so that they are easy to handle. Together with each image we also have the hour of the day (0-23) at which it was taken.

In [ ]:
with open('data/day_night_dataset.pckl', 'rb') as f:
    data = pickle.load(f)

images = data['images']
hours = data['hours']

print('Images', images.shape, images.dtype)
print('Hours ', hours.shape, hours.dtype)

#### Dataset Visualization

Let's now visualize some random data and the corresponding hours to see what we are dealing with here.

In [ ]:
for cnt, idx in enumerate(np.random.randint(0, len(images), 6)):
    plt.subplot(2, 3, cnt + 1), plt.imshow(images[idx])
    plt.title('Hour: ' + str(hours[idx])), plt.axis(False)

#### Data Preprocessing

Now we will prepare the data for the training. We have to turn the hours into binary annotations. We will say that the images taken between 18:00 and 6:00 are night images (label 1) and all the others are day images (label 0).

Be aware that this is a design decision and a bit of an arbitrary one: the images taken close to the sunrise and the sunset (around 6:00 and 18:00) are neither clearly day nor clearly night.

In [ ]:
labels = np.array([1 if (h <= 6 or h >= 18) else 0 for h in hours])

# Are the two classes balanced?
print('Num day images:  ', np.sum(labels == 0))
print('Num night images:', np.sum(labels == 1))
print('Accuracy of always predicting night:', np.mean(labels == 1))

The classes are reasonably balanced, but a classifier that always says *night* is already right in more than half of the cases. That is the bare minimum to beat.

Next, we normalize the images to the range [0, 1] and we split the data into a **training** set (used for fitting the baseline and training the network) and a **test** set (used only for the final comparison). The split is random but **stratified**, which means that both sets keep the same day/night proportions.

In [ ]:
images = images/255

x_train, x_test, y_train, y_test, h_train, h_test = train_test_split(
    images, labels, hours, test_size=0.2, stratify=labels, random_state=1234)

print('Train set:', x_train.shape, '| Test set:', x_test.shape)

#### Build the Baseline

Before we start training our neural network based model, we might need to establish a performance baseline. We do so to be able to tell whether the model is actually good or not.

Since we are dealing with a day-night problem, the most straightforward approach is to use the mean color (or even luminance) to classify images as day vs night. So let's do it :-)

In [ ]:
# Compute the mean color value for each training image
mu_train = np.mean(x_train, axis=(1, 2, 3))

# Visualize the histograms of the obtained mean colors. Are they well separated?
bins = np.linspace(0, 1, 30)
plt.hist(mu_train[y_train == 0], bins=bins, alpha=0.6)
plt.hist(mu_train[y_train == 1], bins=bins, alpha=0.6)
plt.legend(['Day', 'Night']), plt.xlabel('average color'), plt.grid(True)

The histograms do overlap. Anyway, let's find the optimal threshold that maximizes the classification accuracy **on the training set**. Images darker than the threshold will be classified as night.

In [ ]:
thresholds = np.arange(0.01, 1, 0.01)
acc = [np.mean((mu_train < th) == y_train) for th in thresholds]

best_th = thresholds[np.argmax(acc)]
print('Best threshold', best_th, '| Train accuracy', np.max(acc))

plt.plot(thresholds, acc, '.-'), plt.grid(True)
plt.axvline(best_th, color='r', linestyle='--')
plt.xlabel('threshold'), plt.ylabel('accuracy')

Now we apply the threshold to the test set. This number is the baseline our network has to beat.

In [ ]:
mu_test = np.mean(x_test, axis=(1, 2, 3))
pred_baseline = mu_test < best_th

print('Baseline test accuracy', np.mean(pred_baseline == y_test))

#### Binary Classifier Based on Neural Networks

Let's now build a NN-based classifier and let's see if we can beat the baseline. The sigmoid at the output keeps the network output in the range [0, 1], like the labels.

In [ ]:
inputs = Input(shape=(64, 64, 3))

net = Flatten()(inputs)
net = Dense(16, activation='relu')(net)
outputs = Dense(1, activation='sigmoid')(net)

model = Model(inputs, outputs)
model.summary()

Let's compile the model and start training. As in the previous notebook, we use the MSE loss for the sake of simplicity (we will see the proper classification loss in the next lesson). We keep 10 % of the training set aside as a validation set to monitor the training.

In [ ]:
epochs = 25
batch_size = 128

model.compile(loss="mse", optimizer="adam", metrics=["accuracy"])
history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1, verbose=2)

In [ ]:
h = history.history

plt.subplot(121), plt.plot(h['loss'], '.-', h['val_loss'], '.-'), plt.grid(True)
plt.xlabel('epoch'), plt.title('loss'), plt.legend(['Train', 'Validation'])
plt.subplot(122), plt.plot(h['accuracy'], '.-', h['val_accuracy'], '.-'), plt.grid(True)
plt.xlabel('epoch'), plt.title('accuracy'), plt.legend(['Train', 'Validation'])

#### Evaluation

We evaluate the network on the test set, the very same one that was used for the baseline, so that the comparison is fair.

In [ ]:
y_score = model.predict(x_test, verbose=0).flatten()
pred_nn = y_score > 0.5

print('Baseline test acc   ', np.mean(pred_baseline == y_test))
print('Neural net test acc ', np.mean(pred_nn == y_test))
print()
print('Neural net day acc  ', np.mean(pred_nn[y_test == 0] == y_test[y_test == 0]))
print('Neural net night acc', np.mean(pred_nn[y_test == 1] == y_test[y_test == 1]))

Our network does beat the baseline. But where does it still make mistakes? Let's count the errors of both classifiers as a function of the hour of the day.

In [ ]:
errors_nn = np.bincount(h_test[pred_nn != y_test], minlength=24)
errors_baseline = np.bincount(h_test[pred_baseline != y_test], minlength=24)

plt.bar(np.arange(24) - 0.2, errors_baseline, width=0.4)
plt.bar(np.arange(24) + 0.2, errors_nn, width=0.4)
plt.legend(['Baseline', 'Neural net']), plt.grid(True)
plt.xticks(range(24)), plt.xlabel('hour of the day'), plt.ylabel('number of errors')

In our run, almost all the mistakes of the network (17 out of 18) happen at the hours 6 and 17, i.e., exactly where our labelling rule switches between day and night. These are the sunrise and sunset images, where even a human would hesitate and where the label itself is questionable. So the remaining errors are to a large extent a problem of the **data and its labels**, not of the model. The baseline does make these mistakes too, but in addition it makes errors scattered all over the day.

Let's finally look at the misclassified images.

In [ ]:
# Let's look at the mistakes of the network
wrong = np.where(pred_nn != y_test)[0]

for cnt, idx in enumerate(wrong[:10]):
    plt.subplot(2, 5, cnt + 1), plt.imshow(x_test[idx], vmin=0, vmax=1), plt.axis(False)
    plt.title('Hour: ' + str(h_test[idx]) + ' | True: ' + str(y_test[idx]) + ' | Pred: ' + str(int(pred_nn[idx])))

Look at the pictures above: many of the "mistakes" are images from 6:00 that are labelled as night, but clearly show daylight, and images from 17:00 that are labelled as day, but are already dark. In these cases it is arguably the network that is right and the label that is wrong. Lesson learnt: when the accuracy stops improving, always look at the errors, they might tell you more about your data than about your model.